# Research Question 3: False-Positive and False-Negative Subgroup Investigation

**Are false positives/negatives disproportionately associated with particular patient or image characteristics, especially AP/PA view, age, sex, and image acquisition?**

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

In [3]:
from IPython.utils.capture import capture_output

In [4]:
# Import data from the base notebook

# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run r0_dataset_overiew_dashboard.ipynb

In [5]:
HTML_OUTPUT_FOLDER = "html_plots/r3"

## 1 Plot histogram comparing different characteristics against models confidence

In [68]:
MIN_GROUP_SIZE = 20

In [56]:
def show_fp_fn_characteristics(labels_list = XRAY_LABELS_LIST,
                               models_prediction_score = MODEL_SCORE_PREDICTIONS_CONVERTED,
                               showInNotebook = True, output_folder = None,
                               threshold=THRESHOLD):

    meta = XRAY_DATASET.csv.copy().reset_index(drop=True)

    def _first_existing(columns):
        for c in columns:
            if c in meta.columns:
                return c
        return None

    view_col = _first_existing(["AP/PA", "ViewPosition", "View Position", "view"])
    sex_col = _first_existing(["Sex", "Patient Gender", "sex", "Gender"])
    age_col = _first_existing(["Age", "Patient Age", "age"])

    columns = [view_col, sex_col, age_col]

    if view_col:
        meta["View"] = meta[view_col].astype(str).str.strip().str.upper()
        meta["View"] = meta["View"].replace({"NAN": "Unknown", "": "Unknown"})
        meta.loc[~meta["View"].isin(["AP", "PA"]), "View"] = "Other/Unknown"
    else:
        meta["View"] = "Unavailable"

    if sex_col:
        meta["Sex"] = meta[sex_col].astype(str).str.strip().str.upper().replace({
            "M": "Male", "MALE": "Male", "F": "Female", "FEMALE": "Female",
            "NAN": "Unknown", "": "Unknown"
        })
        meta.loc[~meta["Sex"].isin(["Male", "Female"]), "Sex"] = "Unknown"
    else:
        meta["Sex"] = "Unavailable"

    if age_col:
        meta["Age_years"] = pd.to_numeric(meta[age_col], errors="coerce")
        # Keep plausible adult/pediatric ages; do not silently treat sentinel values as real ages.
        meta.loc[(meta["Age_years"] < 0) | (meta["Age_years"] > 120), "Age_years"] = np.nan
        meta["Age_group"] = pd.cut(
            meta["Age_years"], bins=[0, 40, 50, 60, 70, 80, 120],
            labels=["≤40", "41–50", "51–60", "61–70", "71–80", ">80"],
            include_lowest=True
        ).astype("object").fillna("Unknown")
    else:
        meta["Age_years"] = np.nan
        meta["Age_group"] = "Unavailable"

    
    rows = []
    for j, pathology in enumerate(DATASET_PATHOLOGIES):
        truth = np.asarray(labels_list[:, j], dtype=float)
        score = np.asarray(models_prediction_score[:, j], dtype=float)
        valid = np.isfinite(truth) & np.isfinite(score) & np.isin(truth, [0, 1])
        pred = (score >= threshold).astype(int)
        outcome = np.where((truth == 0) & (pred == 1), "False Positive",
                np.where((truth == 1) & (pred == 0), "False Negative",
                np.where((truth == 1) & (pred == 1), "True Positive", "True Negative")))
        tmp = meta.copy()
        tmp["Pathology"] = pathology
        tmp["y_true"] = truth
        tmp["score"] = score
        tmp["y_pred"] = pred
        tmp["Outcome"] = outcome
        rows.append(tmp.loc[valid])

    experiment_df = pd.concat(rows, ignore_index=True)

    def wilson_interval(successes, total, z=1.96):
        if total == 0:
            return np.nan, np.nan
        p = successes / total
        den = 1 + z*z/total
        center = (p + z*z/(2*total)) / den
        half = z * np.sqrt((p*(1-p)/total) + (z*z/(4*total*total))) / den
        return max(0, center-half), min(1, center+half)


    def subgroup_metrics(df, group_col, min_n=MIN_GROUP_SIZE):
        records = []

        for (pathology, group), g in df.groupby(
            ["Pathology", group_col], observed=True, dropna=False
        ):
            negatives = g[g["y_true"] == 0]
            positives = g[g["y_true"] == 1]

            n_neg = len(negatives)
            n_pos = len(positives)

            fp = int((negatives["y_pred"] == 1).sum())
            fn = int((positives["y_pred"] == 0).sum())

            fp_lo, fp_hi = wilson_interval(fp, n_neg)
            fn_lo, fn_hi = wilson_interval(fn, n_pos)

            records.append({
                "Pathology": pathology,
                "Group": str(group),
                "N_images": len(g),

                "N_actual_negative": n_neg,
                "FP": fp,
                "FP_rate": fp / n_neg if n_neg else np.nan,
                "FP_CI_low": fp_lo,
                "FP_CI_high": fp_hi,

                "N_actual_positive": n_pos,
                "FN": fn,
                "FN_rate": fn / n_pos if n_pos else np.nan,
                "FN_CI_low": fn_lo,
                "FN_CI_high": fn_hi,

                "Enough_negative": n_neg >= min_n,
                "Enough_positive": n_pos >= min_n
            })

        return pd.DataFrame(records)

    def plot_error_rates_by_subgroup(group_col, min_n=MIN_GROUP_SIZE):

        stats = subgroup_metrics(experiment_df, group_col, min_n=min_n)

        for error_type in ["FP", "FN"]:

            rate_col = f"{error_type}_rate"
            ci_low = f"{error_type}_CI_low"
            ci_high = f"{error_type}_CI_high"
            count_col = error_type
            denominator = (
                "N_actual_negative" if error_type == "FP"
                else "N_actual_positive"
            )
            enough_col = (
                "Enough_negative" if error_type == "FP"
                else "Enough_positive"
            )

            plot_data = stats[
                stats[enough_col] & stats[rate_col].notna()
            ].copy()

            if plot_data.empty:
                print(
                    f"No groups for {group_col} have at least "
                    f"{min_n} eligible cases for {error_type}."
                )
                continue

            fig = px.bar(
                plot_data,
                x="Pathology",
                y=rate_col,
                color="Group",
                barmode="group",

                error_y=plot_data[ci_high] - plot_data[rate_col],
                error_y_minus=plot_data[rate_col] - plot_data[ci_low],

                hover_data=[
                    "N_images", denominator, count_col,
                    ci_low, ci_high
                ],

                title=(
                    f"{'False-positive' if error_type == 'FP' else 'False-negative'} "
                    f"rate by {group_col} "
                    f"(threshold={threshold}; minimum n={min_n})"
                ),

                labels={
                    rate_col: (
                        "False-positive rate (among actual negatives)"
                        if error_type == "FP"
                        else "False-negative rate (among actual positives)"
                    ),
                    "Pathology": "Finding"
                }
            )

            fig.update_yaxes(tickformat=".0%", range=[0, 1])
            fig.update_layout(
                template="plotly_white",
                height=600,
                width=1250
            )
            if showInNotebook:
                fig.show()
            if output_folder is not None:
                fig.write_html( output_folder + f'/{error_type}_histo_{group_col.replace('/', '_')}_{threshold}.html',
                            include_plotlyjs="cdn",
                            full_html=True)

        return stats

    for col in columns:
        plot_error_rates_by_subgroup(col)

    return experiment_df



In [57]:
show_fp_fn_characteristics(output_folder=HTML_OUTPUT_FOLDER)

,PatientID,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,...,sex_male,sex_female,View,Age_years,Age_group,Pathology,y_true,score,y_pred,Outcome
0,patient05762,CheXpert-v1.0-small/train/patient05762/study1/...,Female,40.0,Frontal,AP,NaN,NaN,NaN,NaN,...,False,True,AP,40.0,≤40,Atelectasis,0.0,0.559698,0,True Negative
1,patient49950,CheXpert-v1.0-small/train/patient49950/study1/...,Male,43.0,Frontal,AP,NaN,NaN,1.0,NaN,...,True,False,AP,43.0,41–50,Atelectasis,0.0,0.859119,1,False Positive
2,patient42246,CheXpert-v1.0-small/train/patient42246/study1/...,Female,66.0,Frontal,AP,NaN,-1.0,1.0,1.0,...,False,True,AP,66.0,61–70,Atelectasis,0.0,0.867330,1,False Positive
3,patient28326,CheXpert-v1.0-small/train/patient28326/study1/...,Female,75.0,Frontal,AP,NaN,NaN,1.0,NaN,...,False,True,AP,75.0,71–80,Atelectasis,0.0,0.966267,1,False Positive
4,patient27948,CheXpert-v1.0-small/train/patient27948/study1/...,Female,52.0,Frontal,PA,NaN,NaN,NaN,-1.0,...,False,True,PA,52.0,51–60,Atelectasis,0.0,0.912938,1,False Positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109995,patient30409,CheXpert-v1.0-small/train/patient30409/study1/...,Female,66.0,Frontal,PA,NaN,NaN,NaN,NaN,...,False,True,PA,66.0,61–70,Pneumothorax,0.0,0.515487,0,True Negative
109996,patient64330,CheXpert-v1.0-small/train/patient64330/study1/...,Female,90.0,Frontal,AP,NaN,NaN,NaN,NaN,...,False,True,AP,90.0,>80,Pneumothorax,0.0,0.502276,0,True Negative
109997,patient22795,CheXpert-v1.0-small/train/patient22795/study1/...,Female,41.0,Frontal,PA,NaN,0.0,NaN,1.0,...,False,True,PA,41.0,41–50,Pneumothorax,0.0,0.530021,0,True Negative
109998,patient36498,CheXpert-v1.0-small/train/patient36498/study4/...,Male,58.0,Frontal,AP,NaN,NaN,NaN,1.0,...,True,False,AP,58.0,51–60,Pneumothorax,0.0,0.285470,0,True Negative


### Observation

**False-positive rate: AP vs PA**: The diagram shows that the false-positive rate for AP is overall higher than for PA. The only exception is Lung lesion where PA is higher.

**False-negative rate: AP VS PA**: Here the diagram shows that PA has a higher false-negative reate overall, but less dramatic as the false positive rate of AP vs PA.

**False-positive rate: male vs female**: The diagrams does not show significant differences between male and female. 

**False-positive rate: age**: The diagrams shows that in particular the age group **>80** has a high false-positive rate. This is especially visible for atelectasis, cardiomegaly, edema, effusion, and lung opacity. The youngest group (≤40) generally has the lowest rates.

Summary: Image view appears to be an important factor associated with model errors. Age top appears to be associated with false-positive errors, with older patients generally experiencing more false alarms.

### Rerun with threshold of 0.5

Run experiment again with a lower threshold as suggested by AI.

In [59]:
show_fp_fn_characteristics(threshold=0.5, output_folder=HTML_OUTPUT_FOLDER)

,PatientID,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,...,sex_male,sex_female,View,Age_years,Age_group,Pathology,y_true,score,y_pred,Outcome
0,patient05762,CheXpert-v1.0-small/train/patient05762/study1/...,Female,40.0,Frontal,AP,NaN,NaN,NaN,NaN,...,False,True,AP,40.0,≤40,Atelectasis,0.0,0.559698,1,False Positive
1,patient49950,CheXpert-v1.0-small/train/patient49950/study1/...,Male,43.0,Frontal,AP,NaN,NaN,1.0,NaN,...,True,False,AP,43.0,41–50,Atelectasis,0.0,0.859119,1,False Positive
2,patient42246,CheXpert-v1.0-small/train/patient42246/study1/...,Female,66.0,Frontal,AP,NaN,-1.0,1.0,1.0,...,False,True,AP,66.0,61–70,Atelectasis,0.0,0.867330,1,False Positive
3,patient28326,CheXpert-v1.0-small/train/patient28326/study1/...,Female,75.0,Frontal,AP,NaN,NaN,1.0,NaN,...,False,True,AP,75.0,71–80,Atelectasis,0.0,0.966267,1,False Positive
4,patient27948,CheXpert-v1.0-small/train/patient27948/study1/...,Female,52.0,Frontal,PA,NaN,NaN,NaN,-1.0,...,False,True,PA,52.0,51–60,Atelectasis,0.0,0.912938,1,False Positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109995,patient30409,CheXpert-v1.0-small/train/patient30409/study1/...,Female,66.0,Frontal,PA,NaN,NaN,NaN,NaN,...,False,True,PA,66.0,61–70,Pneumothorax,0.0,0.515487,1,False Positive
109996,patient64330,CheXpert-v1.0-small/train/patient64330/study1/...,Female,90.0,Frontal,AP,NaN,NaN,NaN,NaN,...,False,True,AP,90.0,>80,Pneumothorax,0.0,0.502276,1,False Positive
109997,patient22795,CheXpert-v1.0-small/train/patient22795/study1/...,Female,41.0,Frontal,PA,NaN,0.0,NaN,1.0,...,False,True,PA,41.0,41–50,Pneumothorax,0.0,0.530021,1,False Positive
109998,patient36498,CheXpert-v1.0-small/train/patient36498/study4/...,Male,58.0,Frontal,AP,NaN,NaN,NaN,1.0,...,True,False,AP,58.0,51–60,Pneumothorax,0.0,0.285470,0,True Negative


### Observation

The observations stay roughly the same as with a threshold of 0.7.

## 2 Age × View analysis

We add a joint age × view analysis to highlight whether these associations remain when the characteristics are considered together.

In [40]:
def age_view_error_rates(long_df, min_n=20):

    records = []

    for (pathology, age, view), g in long_df.groupby(
        ["Pathology", "Age_group", "View"],
        observed=True,
        dropna=False
    ):

        negatives = g[g["y_true"] == 0]
        positives = g[g["y_true"] == 1]

        n_neg = len(negatives)
        n_pos = len(positives)

        fp = (negatives["y_pred"] == 1).sum()
        fn = (positives["y_pred"] == 0).sum()

        records.append({
            "Pathology": pathology,
            "Age_group": age,
            "View": view,

            "N_negative": n_neg,
            "False_positives": fp,
            "FP_rate": fp / n_neg if n_neg else np.nan,

            "N_positive": n_pos,
            "False_negatives": fn,
            "FN_rate": fn / n_pos if n_pos else np.nan,

            "Enough_FP_data": n_neg >= min_n,
            "Enough_FN_data": n_pos >= min_n
        })

    return pd.DataFrame(records)

In [65]:
def plot_age_view_heatmap(stats, pathology, error_type="FN", output_folder=None):

    if error_type == "FN":
        rate_col = "FN_rate"
        enough_col = "Enough_FN_data"
        title = "False-negative rate"
    else:
        rate_col = "FP_rate"
        enough_col = "Enough_FP_data"
        title = "False-positive rate"

    data = stats[
        (stats["Pathology"] == pathology) &
        (stats[enough_col])
    ].copy()

    pivot = data.pivot(
        index="Age_group",
        columns="View",
        values=rate_col
    )

    fig = px.imshow(
        pivot,
        text_auto=".1%",
        color_continuous_scale="Reds",
        zmin=0,
        zmax=1,
        labels={
            "x": "Image view",
            "y": "Age group",
            "color": "Error rate"
        },
        title=f"{title}: {pathology} by Age and View"
    )
    if output_folder is not None:
        fig.write_html(output_folder + f'/age_view_heatmap_{error_type}_{pathology}.html',
                       include_plotlyjs="cdn",
                       full_html=True)
    fig.show()

In [66]:
long_df = show_fp_fn_characteristics(showInNotebook=False)
age_view_stats = age_view_error_rates(long_df)

In [67]:
for pathology in long_df["Pathology"].unique():
    plot_age_view_heatmap(age_view_stats, pathology, "FP", output_folder=HTML_OUTPUT_FOLDER)

### Observation

The generated heatmaps show clear age- and view-related pattern in false-positive rates for Atelectasis, Edema, Lung Opacity and Cardiomegagaly. These findings suggest that both patient age and imaging view are associated with the model's tendency to incorrectly identify these diseases.

The AI suggested run Run a regression analysis tp test whether age, view, and their interaction remain associated with false-positive predictions when considered together. We stopped the experiment here as regression analysis is outside the scope of this lab.